# 01 - Synthetic data generation
**Synthetic data is for model development and integration testing only; it must not be read as real YouTube audience behavior.**
Generates `synthetic_v1` (420 gaming videos, 10-15 min) with a transparent retention simulator (see `docs/synthetic-data.md`).

In [ ]:
# --- Colab / local bootstrap -------------------------------------------------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not os.path.exists("src"):
    # Colab: upload/clone the repository first, e.g.  !git clone <your repo url> retainai && %cd retainai
    raise SystemExit("Clone or upload the repository so that the `src/` folder is in the working directory.")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
root = os.getcwd()
while root != os.path.dirname(root) and not os.path.isdir(os.path.join(root, "src")):
    root = os.path.dirname(root)
os.chdir(root); sys.path.insert(0, root)
import torch
print("repo root:", root, "| GPU:", torch.cuda.is_available())

In [ ]:
from src.data_generation.generate import generate_dataset
from src.data_generation.stats import dataset_stats
import json, os
from src import config
if not (config.synthetic_dir() / 'manifest.csv').exists():
    print(generate_dataset(seed=config.SEED))
print(json.dumps(dataset_stats(), indent=2))

In [ ]:
from src.data_generation import show_sample  # prints one transcript head + its retention curve


In [ ]:
import matplotlib.pyplot as plt, numpy as np
from src.data_generation.stats import load_retention, load_manifest
ret, man = load_retention(), load_manifest()
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, grp in man.groupby('archetype'):
    v = ret[grp.video_id.iloc[0]]['values']; ax[0].plot(np.linspace(0, 1, len(v)), v, label=a)
ax[0].set_title('One synthetic retention curve per archetype'); ax[0].set_xlabel('relative position'); ax[0].set_ylabel('retention %'); ax[0].legend(fontsize=6)
ax[1].hist(man.end_retention, bins=25); ax[1].set_title('Final retention (synthetic)')
plt.show()